In [ ]:
import yfinance as yf
import sqlite3
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sb

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.svm import SVC
from xgboost import XGBClassifier
from sklearn import metrics

import warnings
warnings.filterwarnings('ignore')

In [ ]:
DB_DIR = os.path.join(
    '/Users/aliemre2023/Desktop/apache_project/bist100_airflow/src/db'
)
DB_PATH = os.path.join(DB_DIR, 'bist.db')

def get_connection() -> sqlite3.Connection:
    os.makedirs(DB_DIR, exist_ok=True)
    conn = sqlite3.connect(DB_PATH)
    conn.row_factory = sqlite3.Row
    conn.execute("PRAGMA foreign_keys = ON")
    return conn

conn = get_connection()

In [ ]:
SIRKET_CODE = "AAPL"

In [ ]:
hist = yf.Ticker(f"{SIRKET_CODE}").history(period="2000d")

In [ ]:
hist[:5]

In [ ]:
df = hist
df.shape

In [ ]:
df.describe()

In [ ]:
df.info()

In [ ]:
plt.figure(figsize=(15,5))
plt.plot(df['Close'])
plt.title(f'{SIRKET_CODE} Close price.', fontsize=15)
plt.ylabel('Price in tl.')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
hist = df[-5:]
ax.plot(hist.index, hist["Close"], color="tab:blue", marker="o", label="Close")
ax.fill_between(hist.index, hist["Low"], hist["High"], alpha=0.2, color="tab:blue", label="Low–High Range")

ax.set_xlabel("Date")
ax.set_ylabel("Price (₺)")
ax.set_title(f"{SIRKET_CODE}.IS")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m-%d"))
ax.legend()
fig.autofmt_xdate()
fig.tight_layout()
plt.show()

In [ ]:
df.isnull().sum()

In [ ]:
features = ['Open', 'High', 'Low', 'Close', 'Volume']

plt.subplots(figsize=(20,10))

for i, col in enumerate(features):
  plt.subplot(2,3,i+1)
  sb.distplot(df[col])
plt.show()

In [ ]:
plt.subplots(figsize=(20,10))
for i, col in enumerate(features):
  plt.subplot(2,3,i+1)
  sb.boxplot(df[col])
plt.show()

In [ ]:
df.columns

In [ ]:
df

In [ ]:
df['day'] = df.index.day
df['month'] = df.index.month
df['year'] = df.index.year

df.head()

In [ ]:
df['is_quarter_end'] = np.where(df['month']%3==0,1,0)
df[df["is_quarter_end"] == 1].head()

In [ ]:
data_grouped = df.groupby('year')[['Open', 'High', 'Low', 'Close']].mean()

plt.subplots(figsize=(20, 10))
for i, col in enumerate(['Open', 'High', 'Low', 'Close']):
    plt.subplot(2, 2, i + 1)
    data_grouped[col].plot.bar()
    plt.title(col)
plt.tight_layout()
plt.show()

In [ ]:
df.groupby('is_quarter_end').mean()

In [ ]:
import requests

API_KEY = os.getenv("EVDS_API_KEY", "xxx")

url = "https://evds3.tcmb.gov.tr/igmevdsms-dis/series=TP.DK.USD.A-TP.DK.EUR.A-TP.DK.CHF.ATP.DK.GBP.ATP.DK.JPY.A&startDate=01-10-2017&endDate=01-11-2017&type=json"

# Farklı header formatlarını dene
test_cases = [
    {"key": API_KEY}
]

for headers in test_cases:
    r = requests.get(url, headers=headers)
    print(r)
    is_json = r.text.strip().startswith('[') or r.text.strip().startswith('{')
    print(is_json)
    header_name = list(headers.keys())[0]
    print(f"{header_name}: status={r.status_code}, JSON={is_json}, ilk50={r.text[:]}")
    if is_json:
        print(f"✅ ÇALIŞIYOR! Header: {header_name}")
        break


In [ ]:
import requests
import pandas as pd
from io import StringIO
from datetime import datetime
from typing import Optional

In [ ]:
SERIES_CONFIG = {
    # ── Döviz Kurları ──────────────────────────────────────────────────────
    "usd_buy":        ("TP.DK.USD.A.YTL",  "USD Alış (TRY)"),
    "usd_sell":       ("TP.DK.USD.S.YTL",  "USD Satış (TRY)"),
    "eur_buy":        ("TP.DK.EUR.A.YTL",  "EUR Alış (TRY)"),
    "eur_sell":       ("TP.DK.EUR.S.YTL",  "EUR Satış (TRY)"),
    "gbp_buy":        ("TP.DK.GBP.A.YTL",  "GBP Alış (TRY)"),
    "chf_buy":        ("TP.DK.CHF.A.YTL",  "CHF Alış (TRY)"),
    "jpy_buy":        ("TP.DK.JPY.A.YTL",  "JPY Alış (TRY)"),

    # ── Altın ──────────────────────────────────────────────────────────────
    "gold_try":       ("TP.DK.XAU.A.YTL",  "Altın (TRY/ons)"),   # gram altın için farklı seri gerekebilir
    "gold_usd":       ("TP.DK.XAU.A",      "Altın (USD/ons)"),

    # ── Faiz Oranları ──────────────────────────────────────────────────────
    "policy_rate":    ("TP.MB.B.C",         "TCMB Politika Faizi"),
    "overnight_borrow": ("TP.MB.B.A",       "Gecelik Borçlanma Faizi"),
    "overnight_lend": ("TP.MB.B.B",         "Gecelik Borç Verme Faizi"),

    # ── Enflasyon / TÜFE ───────────────────────────────────────────────────
    "cpi_total":      ("TP.FG.J0",          "TÜFE Genel (Aylık % Değişim)"),
    "cpi_yoy":        ("TP.FG.J01",         "TÜFE Genel (Yıllık % Değişim)"),
    "ppi_total":      ("TP.FG.J02",         "ÜFE Genel (Aylık % Değişim)"),

    # ── Dış Ticaret / Rezervler ────────────────────────────────────────────
    "exports":        ("TP.DIE.IHRIX01",    "İhracat (Milyon USD)"),
    "imports":        ("TP.DIE.ITHALAT",    "İthalat (Milyon USD)"),
    "fx_reserves":    ("TP.AB.B1",          "Brüt Döviz Rezervleri (Milyon USD)"),
    "gold_reserves":  ("TP.AB.B3",          "Altın Rezervleri (Milyon USD)"),
    "current_account":("TP.OB.CA",          "Cari Denge (Milyon USD)"),
}


# Frekans grupları — aynı frekanstaki seriler tek çağrıda çekilir
FREQ_GROUPS = {
    "daily":   ["usd_buy", "usd_sell", "eur_buy", "eur_sell",
                "gbp_buy", "chf_buy", "jpy_buy",
                "gold_try", "gold_usd"],
    "monthly": ["policy_rate", "overnight_borrow", "overnight_lend",
                "cpi_total", "cpi_yoy", "ppi_total",
                "exports", "imports",
                "fx_reserves", "gold_reserves", "current_account"],
}

BASE_URL = "https://evds3.tcmb.gov.tr/igmevdsms-dis/series={series}" \
           "&startDate={start}&endDate={end}&type=json" \
           "&aggregationTypes={agg}&formulas={formula}&frequency={freq}"



In [ ]:

class EVDSCollector:
    """TCMB EVDS API'den trading feature'ları çeken sınıf."""

    def __init__(self, api_key: str):
        """
        Parameters
        ----------
        api_key : str
            EVDS profilinizden aldığınız API anahtarı.
        """
        self.api_key = api_key
        self.session = requests.Session()
        self.session.headers.update({"key": self.api_key})

    # ── Düşük seviye: ham JSON çek ─────────────────────────────────────────

    def _fetch_raw(
        self,
        series_codes: list[str],
        start: str,
        end: str,
        freq: int = 1,       # 1=günlük, 5=aylık
        agg: str = "avg",
        formula: int = 0,    # 0=düzey
    ) -> dict:
        """EVDS'den ham JSON döndürür."""
        joined = "-".join(series_codes)
        n = len(series_codes)
        agg_str     = "-".join([agg] * n)
        formula_str = "-".join([str(formula)] * n)

        url = BASE_URL.format(
            series=joined,
            start=start,
            end=end,
            agg=agg_str,
            formula=formula_str,
            freq=freq,
        )
        resp = self.session.get(url, timeout=30)
        resp.raise_for_status()
        return resp.json()

    # ── JSON → DataFrame ───────────────────────────────────────────────────

    @staticmethod
    def _json_to_df(raw: dict, rename_map: dict) -> pd.DataFrame:
        """
        EVDS JSON çıktısını temiz bir DataFrame'e dönüştürür.

        Parameters
        ----------
        raw        : EVDS JSON yanıtı
        rename_map : {EVDS_seri_kodu: kolay_isim}
        """
        items = raw.get("items", [])
        if not items:
            return pd.DataFrame()

        df = pd.DataFrame(items)

        # Tarih sütunu her zaman "Tarih" adıyla gelir
        if "Tarih" not in df.columns:
            return df

        df["date"] = pd.to_datetime(df["Tarih"], dayfirst=True, errors="coerce")
        df = df.drop(columns=["Tarih"], errors="ignore")
        df = df.set_index("date").sort_index()

        # Sütunları yeniden adlandır ve sayısala çevir
        df = df.rename(columns=rename_map)
        for col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

        return df

    # ── Grup bazlı çekme ───────────────────────────────────────────────────

    def _fetch_group(
        self,
        group: str,           # "daily" | "monthly"
        start: str,
        end: str,
    ) -> pd.DataFrame:
        """Belirtilen gruptaki tüm serileri çeker ve birleştirir."""
        keys = FREQ_GROUPS[group]
        freq = 1 if group == "daily" else 5
        agg  = "avg"

        codes      = [SERIES_CONFIG[k][0] for k in keys]
        rename_map = {SERIES_CONFIG[k][0]: k for k in keys}

        print(f"  [{group}] {len(codes)} seri çekiliyor...")
        try:
            raw = self._fetch_raw(codes, start, end, freq=freq, agg=agg)
            df  = self._json_to_df(raw, rename_map)
            print(f"  [{group}] ✓ {len(df)} satır, {len(df.columns)} sütun")
            return df
        except Exception as e:
            print(f"  [{group}] ✗ HATA: {e}")
            return pd.DataFrame()

    # ── Feature Engineering ────────────────────────────────────────────────

    @staticmethod
    def add_derived_features(df: pd.DataFrame) -> pd.DataFrame:
        """
        Ham veriden trading için türetilmiş feature'lar ekler.

        Eklenen feature'lar
        -------------------
        - usd_spread         : USD satış - alış farkı (likidite/stres göstergesi)
        - eur_usd_cross      : EUR/USD çapraz kur
        - gold_try_pct       : Altın TRY günlük % değişim
        - usd_try_pct        : USD/TRY günlük % değişim
        - usd_try_ma5/ma20   : 5 ve 20 günlük hareketli ortalama
        - usd_try_vol10      : 10 günlük realized volatilite
        - real_rate          : Reel faiz (politika faizi - yıllık TÜFE)
        - trade_balance      : İhracat - İthalat
        - fx_gold_total_res  : Toplam rezerv (döviz + altın)
        """
        d = df.copy()

        # Döviz spread & çapraz kurlar
        if {"usd_buy", "usd_sell"}.issubset(d.columns):
            d["usd_spread"] = d["usd_sell"] - d["usd_buy"]

        if {"eur_buy", "usd_buy"}.issubset(d.columns):
            d["eur_usd_cross"] = d["eur_buy"] / d["usd_buy"]

        # Fiyat değişimleri
        if "gold_try" in d.columns:
            d["gold_try_pct"] = d["gold_try"].pct_change() * 100

        if "usd_buy" in d.columns:
            d["usd_try_pct"]  = d["usd_buy"].pct_change() * 100
            d["usd_try_ma5"]  = d["usd_buy"].rolling(5).mean()
            d["usd_try_ma20"] = d["usd_buy"].rolling(20).mean()
            d["usd_try_vol10"]= d["usd_buy"].pct_change().rolling(10).std() * 100

        # Reel faiz (aylık serilerin günlük index'e forward-fill edilmiş hali)
        if {"policy_rate", "cpi_yoy"}.issubset(d.columns):
            d["real_rate"] = d["policy_rate"].ffill() - d["cpi_yoy"].ffill()

        # Dış ticaret dengesi
        if {"exports", "imports"}.issubset(d.columns):
            d["trade_balance"] = d["exports"] - d["imports"]

        # Toplam rezerv
        if {"fx_reserves", "gold_reserves"}.issubset(d.columns):
            d["fx_gold_total_res"] = d["fx_reserves"] + d["gold_reserves"]

        return d

    # ── Ana metot ──────────────────────────────────────────────────────────

    def get_all_features(
        self,
        start: str = "01-01-2015",
        end: str   = "01-01-2999",
        add_derived: bool = True,
        resample_freq: Optional[str] = None,   # örn: "W", "M", "Q"
    ) -> pd.DataFrame:
        """
        Tüm gruplardan veri çeker, birleştirir ve isteğe bağlı türetilmiş
        feature'lar ekler.

        Parameters
        ----------
        start         : Başlangıç tarihi (gg-aa-yyyy)
        end           : Bitiş tarihi (gg-aa-yyyy)
        add_derived   : Türetilmiş feature'lar eklensin mi?
        resample_freq : Veriyi yeniden örnekle (None=dokunma)

        Returns
        -------
        pd.DataFrame  : Date-indexed birleşik feature DataFrame
        """
        print(f"\n{'='*55}")
        print(f"  EVDS Feature Collection")
        print(f"  {start}  →  {end}")
        print(f"{'='*55}")

        frames = []
        for group in FREQ_GROUPS:
            df_g = self._fetch_group(group, start, end)
            if not df_g.empty:
                frames.append(df_g)

        if not frames:
            print("⚠  Hiç veri çekilemedi.")
            return pd.DataFrame()

        # Outer join — farklı frekanslar birbirini bozmaz
        combined = pd.concat(frames, axis=1, join="outer").sort_index()

        if add_derived:
            combined = self.add_derived_features(combined)
            print(f"\n  ✓ Türetilmiş feature'lar eklendi.")

        if resample_freq:
            combined = combined.resample(resample_freq).last()
            print(f"  ✓ {resample_freq} frekansına resampled.")

        print(f"\n  Sonuç: {len(combined)} satır × {len(combined.columns)} sütun")
        print(f"{'='*55}\n")
        return combined

    # ── Tekil seri çekme ──────────────────────────────────────────────────

    def get_series(
        self,
        series_code: str,
        start: str,
        end: str,
        freq: int = 1,
        agg: str = "avg",
        formula: int = 0,
    ) -> pd.Series:
        """
        Tek bir EVDS seri kodunu pd.Series olarak döndürür.

        Örnek
        -----
        s = collector.get_series("TP.DK.USD.A.YTL", "01-01-2023", "01-01-2999")
        """
        raw = self._fetch_raw([series_code], start, end, freq, agg, formula)
        df  = self._json_to_df(raw, {series_code: "value"})
        return df["value"] if "value" in df.columns else pd.Series(dtype=float)



In [ ]:
API_KEY = os.getenv("EVDS_API_KEY", "xxx")

collector = EVDSCollector(api_key=API_KEY)

In [ ]:
import pandas as pd

chunks = []
years = range(2018, 2027)

for y in years:
    start = f"01-01-{y}"
    end   = f"31-12-{y}"
    try:
        chunk = collector.get_all_features(start=start, end=end, add_derived=False)
        chunks.append(chunk)
        print(f"{y}: {len(chunk)} satır")
    except Exception as e:
        print(f"{y}: HATA - {e}")

df_evds = pd.concat(chunks).sort_index()
df_evds = df_evds[~df_evds.index.duplicated(keep='last')]
print(f"Toplam EVDS: {df_evds.shape}")

In [ ]:
evds_cols = [
    'TP_DK_USD_A_YTL',
    'TP_DK_USD_S_YTL', 
    'TP_DK_EUR_A_YTL',
    'TP_DK_EUR_S_YTL',
    'TP_DK_GBP_A_YTL',
    'TP_DK_CHF_A_YTL',
    'TP_DK_JPY_A_YTL',

    'TP_FG_J0',
    'TP_FG_J01',
    'TP_FG_J02',
    'TP_AB_B1',
    'TP_AB_B3'

    
]

In [ ]:
print(df.index.tz)      
print(df_evds.index.tz)

In [ ]:
df.index = df.index.tz_convert(None)
df.index = df.index.normalize()

In [ ]:
df_evds.head()

In [ ]:
df.head()

In [ ]:
# ── Makro verileri ana df'ye join et ──
df = df.join(df_evds[evds_cols], how='left')

# Forward fill (hafta sonu / tatil günleri için)
df['TP_DK_USD_A_YTL'].fillna(method='ffill', inplace=True)
df['TP_DK_USD_S_YTL'].fillna(method='ffill', inplace=True)
df['TP_DK_EUR_A_YTL'].fillna(method='ffill', inplace=True)
df['TP_DK_EUR_S_YTL'].fillna(method='ffill', inplace=True)
df['TP_DK_GBP_A_YTL'].fillna(method='ffill', inplace=True)
df['TP_DK_CHF_A_YTL'].fillna(method='ffill', inplace=True)
df['TP_DK_JPY_A_YTL'].fillna(method='ffill', inplace=True)

# Forward fill (aylık değerler)
df['TP_FG_J0'].fillna(method='ffill', inplace=True)
df['TP_FG_J01'].fillna(method='ffill', inplace=True)
df['TP_FG_J02'].fillna(method='ffill', inplace=True)
df['TP_AB_B1'].fillna(method='ffill', inplace=True)
df['TP_AB_B3'].fillna(method='ffill', inplace=True)

# ── Döviz Kuru Feature'ları ──
df['USD_spread']      = df['TP_DK_USD_S_YTL'] - df['TP_DK_USD_A_YTL']           # Alış-satış farkı
df['USD_change']      = df['TP_DK_USD_A_YTL'].pct_change()               # Günlük dolar değişimi
df['USD_change_5d']   = df['TP_DK_USD_A_YTL'].pct_change(5)              # 5 günlük dolar değişimi
df['USD_sma_7']       = df['TP_DK_USD_A_YTL'].rolling(7).mean()          # 7 günlük USD SMA
df['USD_volatility']  = df['TP_DK_USD_A_YTL'].pct_change().rolling(14).std()  # Dolar volatilitesi
df['EUR_USD_ratio']   = df['TP_DK_EUR_A_YTL'] / df['TP_DK_USD_A_YTL']            # EUR/USD oranı

# ── Fiyat / Dolar Oranı (TL bazlı şirketler için önemli) ──
df['close_usd'] = df['Close'] / df['TP_DK_USD_A_YTL']                    # Dolar bazlı fiyat
df['close_usd_change'] = df['close_usd'].pct_change()            # Dolar bazlı getiri

# ── Faiz Feature'ları ──
#df['rate_change'] = df['policy_rate'].diff()                     # Faiz değişimi
#df['real_rate']   = df['policy_rate'] - (df['USD_change'].rolling(30).mean() * 100)  # Yaklaşık reel faiz

# NaN temizle
df.dropna(inplace=True)

print(f"Yeni df shape: {df.shape}")
df[['TP_DK_USD_A_YTL', 'USD_change', 'EUR_USD_ratio', 'close_usd']].tail()

In [ ]:
df.tail()

In [ ]:
query = """
SELECT n.news_date AS date,
       AVG(n.news_ratio) AS avg_sentiment,
       COUNT(n.news_id) AS news_count
FROM   news n
JOIN   news_sirket ns ON ns.news_id  = n.news_id
JOIN   sirket      s  ON s.sirket_id = ns.sirket_id
WHERE  s.sirket_code = ?
GROUP BY n.news_date
ORDER BY n.news_date
"""

sentiment_df = pd.read_sql_query(query, conn, params=(SIRKET_CODE,))

sentiment_df['date'] = pd.to_datetime(sentiment_df['date'])
sentiment_df.set_index('date', inplace=True)

if hasattr(df.index, 'tz') and df.index.tz is not None:
    df.index = df.index.tz_localize(None)

df = df.join(sentiment_df, how='left')

df['avg_sentiment'].fillna(0, inplace=True)
df['news_count'].fillna(0, inplace=True)

# Son 7 günün ortalama sentiment'i
df['sentiment_7d'] = df['avg_sentiment'].rolling(window=7).mean().fillna(0)
df['sentiment_14d'] = df['avg_sentiment'].rolling(window=14).mean().fillna(0)
df['sentiment_30d'] = df['avg_sentiment'].rolling(window=30).mean().fillna(0)
df['sentiment_50d'] = df['avg_sentiment'].rolling(window=50).mean().fillna(0)

In [ ]:
df['open-close']  = df['Open'] - df['Close']
df['low-high']  = df['Low'] - df['High']

# Teknik indicators

# SMA (Simple Moving Average)
df['SMA_7']  = df['Close'].rolling(window=7).mean()
df['SMA_21'] = df['Close'].rolling(window=21).mean()
df['SMA_50'] = df['Close'].rolling(window=50).mean()

# EMA (Exponential Moving Average)
df['EMA_12'] = df['Close'].ewm(span=12, adjust=False).mean()
df['EMA_26'] = df['Close'].ewm(span=26, adjust=False).mean()

# MACD
df['MACD']        = df['EMA_12'] - df['EMA_26']
df['MACD_signal'] = df['MACD'].ewm(span=9, adjust=False).mean()
df['MACD_hist']   = df['MACD'] - df['MACD_signal']

# RSI (Relative Strength Index) — 14 günlük
delta = df['Close'].diff()
gain  = delta.where(delta > 0, 0).rolling(window=14).mean()
loss  = (-delta.where(delta < 0, 0)).rolling(window=14).mean()
rs    = gain / loss
df['RSI_14'] = 100 - (100 / (1 + rs))

# Bollinger Bands (20 günlük)
df['BB_mid']   = df['Close'].rolling(window=20).mean()
df['BB_std']   = df['Close'].rolling(window=20).std()
df['BB_upper'] = df['BB_mid'] + 2 * df['BB_std']
df['BB_lower'] = df['BB_mid'] - 2 * df['BB_std']
df['BB_width'] = df['BB_upper'] - df['BB_lower']

# ATR (Average True Range) — 14 günlük
high_low   = df['High'] - df['Low']
high_close = (df['High'] - df['Close'].shift(1)).abs()
low_close  = (df['Low']  - df['Close'].shift(1)).abs()
true_range = pd.concat([high_low, high_close, low_close], axis=1).max(axis=1)
df['ATR_14'] = true_range.rolling(window=14).mean()

# Stochastic Oscillator (%K, %D) — 14 günlük
low_14  = df['Low'].rolling(window=14).min()
high_14 = df['High'].rolling(window=14).max()
df['Stoch_K'] = 100 * (df['Close'] - low_14) / (high_14 - low_14)
df['Stoch_D'] = df['Stoch_K'].rolling(window=3).mean()

# OBV (On-Balance Volume)
df['OBV'] = (np.sign(df['Close'].diff()) * df['Volume']).fillna(0).cumsum()

# Momentum
df['Momentum_10'] = df['Close'] - df['Close'].shift(10)

# VWAP (Volume Weighted Average Price — günlük yaklaşık)
df['VWAP'] = (df['Volume'] * (df['High'] + df['Low'] + df['Close']) / 3).cumsum() / df['Volume'].cumsum()

# Rolling ile oluşan NaN'ları temizle
df.dropna(inplace=True)

In [ ]:
!/usr/local/bin/python3.10 -m pip install fredapi

In [ ]:
from fredapi import Fred
import pandas as pd

API_KEY = os.getenv("FRED_API_KEY", "xxx")
fred = Fred(api_key=API_KEY)

# Stock prediction için en değerli günlük seriler
seriler = {
    "FEDFUNDS"  : "Fed Faiz Oranı",
    "DGS10"     : "10Y Tahvil Faizi",
    "DGS2"      : "2Y Tahvil Faizi",
    "T10Y2Y"    : "Yield Curve (10Y-2Y spread)",
    "VIXCLS"    : "VIX (volatilite endeksi)",
    "DCOILWTICO": "Ham Petrol Fiyatı",
    "DEXUSEU"   : "EUR/USD",
    "DEXCHUS"   : "USD/CNY",
    "UMCSENT"   : "Tüketici Güven Endeksi",
    "TEDRATE"   : "TED Spread (kredi riski)",
}

df_fred = pd.DataFrame()
for kod, isim in seriler.items():
    seri = fred.get_series(kod, observation_start="2015-01-01")
    df_fred[isim] = seri
    print(f"✅ {isim}")

df_fred = df_fred.sort_index().ffill()
print(df_fred.tail())

In [ ]:
df_fred.head()

In [ ]:
df_fred.columns

In [ ]:
df_fred = df_fred.drop(columns=["EUR/USD"])
df_fred.columns

In [ ]:
df = df.join(df_fred, how='left')

In [ ]:
fred_cols = [
    'Fed Faiz Oranı', '10Y Tahvil Faizi', '2Y Tahvil Faizi',
    'Yield Curve (10Y-2Y spread)', 'VIX (volatilite endeksi)',
    'Ham Petrol Fiyatı', 'USD/CNY', 'Tüketici Güven Endeksi',
    'TED Spread (kredi riski)'
]

In [ ]:
df[fred_cols] = df[fred_cols].ffill().bfill()

In [ ]:
print(df[fred_cols].isna().sum())

In [ ]:
df.head()

In [ ]:
'''
url = "https://api.worldbank.org/v2/country/TR/indicator/FP.CPI.TOTL.ZG?format=json&per_page=10"
response = requests.get(url)
data = response.json()

records = []
for item in data[1]:
    records.append({
        "Yıl": item["date"],
        "Enflasyon (%)": item["value"]
    })

df = pd.DataFrame(records)
df = df.dropna()
df = df.sort_values("Yıl")
print(df.to_string(index=False))
'''

In [ ]:
df['target'] = np.where(df['Close'].shift(-1) > df['Close'], 1, 0) # buy dates

In [ ]:
df.head()

In [ ]:
plt.pie(df['target'].value_counts().values, 
        labels=[0, 1], autopct='%1.1f%%')
plt.show()

In [ ]:
plt.figure(figsize=(10, 10)) 

# As our concern is with the highly 
# correlated features only so, we will visualize 
# our heatmap as per that criteria only. 
sb.heatmap(df.corr() > 0.9, annot=True, cbar=False)
plt.show()

In [ ]:
features = df[['open-close', 'low-high', 'is_quarter_end',
               'SMA_7', 'SMA_21', 'SMA_50',
               'MACD', 'MACD_signal', 'MACD_hist',
               'RSI_14',
               'BB_width', 'BB_upper', 'BB_lower',
               'ATR_14',
               'Stoch_K', 'Stoch_D',
               'OBV', 'Momentum_10', 'VWAP',
               # Sentiment
               'avg_sentiment', 'news_count', 'sentiment_7d',
               'sentiment_14d', 'sentiment_30d',

               # Makroekonomik (TCMB) # EVDS API
               'USD_change', 'USD_change_5d',
               'USD_sma_7', 
               'close_usd', 'close_usd_change',

               "TP_DK_USD_S_YTL", 'TP_DK_USD_A_YTL', 
               "USD_spread", "USD_volatility", "EUR_USD_ratio",

               "TP_FG_J0", "TP_FG_J01", "TP_FG_J02",
               "TP_AB_B1", "TP_AB_B3",

               # ABD bilgileri FRED
               'Fed Faiz Oranı', '10Y Tahvil Faizi', '2Y Tahvil Faizi',
               'Yield Curve (10Y-2Y spread)', 'VIX (volatilite endeksi)',
               'Ham Petrol Fiyatı', 'USD/CNY', 'Tüketici Güven Endeksi',
               'TED Spread (kredi riski)',
               ]]



target = df['target']

scaler = StandardScaler()
features = scaler.fit_transform(features)

X_train, X_valid, Y_train, Y_valid = train_test_split(
    features, 
    target, 
    test_size=0.1, 
    random_state=42, 
    stratify=target
)
print(X_train.shape, X_valid.shape)

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
from sklearn import metrics


class TradingNN(nn.Module):
    """
    Tabular data için Residual Block'lu NN.

    Mimari:
        Input → Block1(256) → Block2(128) → Block3(64) → Output(1)

    Her blok:
        Linear → BatchNorm → GELU → Dropout + Residual bağlantı
    """

    def __init__(self, input_dim: int, dropout: float = 0.3):
        super().__init__()

        self.input_proj = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.BatchNorm1d(256),
            nn.GELU(),
            nn.Dropout(dropout),
        )

        self.block1 = self._res_block(256, 256, dropout)
        self.block2 = self._res_block(256, 128, dropout)
        self.block3 = self._res_block(128, 64,  dropout)

        self.head = nn.Sequential(
            nn.Linear(64, 32),
            nn.GELU(),
            nn.Linear(32, 1),
        )

    @staticmethod
    def _res_block(in_dim, out_dim, dropout):
        block    = nn.Sequential(
            nn.Linear(in_dim, out_dim),
            nn.BatchNorm1d(out_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        shortcut = nn.Linear(in_dim, out_dim) if in_dim != out_dim else nn.Identity()
        return nn.ModuleList([block, shortcut])

    def _forward_res(self, x, res_block):
        block, shortcut = res_block
        return block(x) + shortcut(x)

    def forward(self, x):
        x = self.input_proj(x)
        x = self._forward_res(x, self.block1)
        x = self._forward_res(x, self.block2)
        x = self._forward_res(x, self.block3)
        return self.head(x).squeeze(1)



def train_trading_nn(
    X_train, Y_train,
    X_valid, Y_valid,
    epochs=100,
    batch_size=256,
    lr=1e-3,
    dropout=0.3,
    patience=15,
    device=None,
):
    if device is None:
        device = ("cuda" if torch.cuda.is_available()
                  else "mps" if torch.backends.mps.is_available()
                  else "cpu")
    print(f"Device: {device}")

    def to_tensor(X, y):
        return (
            torch.tensor(X, dtype=torch.float32).to(device),
            torch.tensor(
                y.values if hasattr(y, 'values') else np.array(y),
                dtype=torch.float32
            ).to(device),
        )

    Xt, Yt = to_tensor(X_train, Y_train)
    Xv, Yv = to_tensor(X_valid, Y_valid)

    loader = DataLoader(
        TensorDataset(Xt, Yt),
        batch_size=batch_size,
        shuffle=True,
        drop_last=True,
    )

    model     = TradingNN(input_dim=X_train.shape[1], dropout=dropout).to(device)
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    # Dengesiz sınıflar için ağırlıklı loss
    y_arr      = Y_train.values if hasattr(Y_train, 'values') else np.array(Y_train)
    pos_weight = torch.tensor((y_arr == 0).sum() / max((y_arr == 1).sum(), 1),
                              dtype=torch.float32).to(device)
    criterion  = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    history    = {"train_auc": [], "val_auc": [], "train_loss": []}
    best_auc   = 0.0
    best_state = None
    no_improve = 0

    print(f"\n{'Epoch':>6} | {'Loss':>8} | {'Train AUC':>9} | {'Val AUC':>8}")
    print("-" * 42)

    for epoch in range(1, epochs + 1):
        model.train()
        total_loss = 0.0
        for xb, yb in loader:
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            total_loss += loss.item()
        scheduler.step()

        model.eval()
        with torch.no_grad():
            train_prob = torch.sigmoid(model(Xt)).cpu().numpy()
            val_prob   = torch.sigmoid(model(Xv)).cpu().numpy()

        train_auc = metrics.roc_auc_score(y_arr, train_prob)
        val_auc   = metrics.roc_auc_score(
            Y_valid.values if hasattr(Y_valid, 'values') else np.array(Y_valid),
            val_prob
        )
        avg_loss  = total_loss / len(loader)

        history["train_auc"].append(train_auc)
        history["val_auc"].append(val_auc)
        history["train_loss"].append(avg_loss)

        if epoch % 10 == 0 or epoch == 1:
            print(f"{epoch:>6} | {avg_loss:>8.4f} | {train_auc:>9.4f} | {val_auc:>8.4f}")

        if val_auc > best_auc:
            best_auc   = val_auc
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            no_improve = 0
        else:
            no_improve += 1
            if no_improve >= patience:
                print(f"\n⏹  Early stopping @ epoch {epoch} | Best Val AUC: {best_auc:.4f}")
                break

    model.load_state_dict(best_state)
    print(f"\n✓ Tamamlandı. Best Val AUC: {best_auc:.4f}")
    return model, history


def evaluate(model, X, Y, label="", device=None):
    if device is None:
        device = next(model.parameters()).device

    model.eval()
    with torch.no_grad():
        prob = torch.sigmoid(
            model(torch.tensor(X, dtype=torch.float32).to(device))
        ).cpu().numpy()

    pred = (prob >= 0.5).astype(int)
    Y_np = Y.values if hasattr(Y, 'values') else np.array(Y)

    print(f"\n── {label} ──")
    print(f"AUC      : {metrics.roc_auc_score(Y_np, prob):.4f}")
    print(f"Accuracy : {metrics.accuracy_score(Y_np, pred):.4f}")
    print(metrics.classification_report(Y_np, pred, target_names=["Down", "Up"]))
    return prob


In [ ]:
class TradingNNWrapper:
    """sklearn-compatible wrapper — predict_proba döndürür."""
    
    def __init__(self, **kwargs):
        self.kwargs = kwargs
        self.model  = None

    def fit(self, X, y, X_val=None, y_val=None):
        self.model, self.history = train_trading_nn(
            X, y, X_val, y_val, **self.kwargs
        )
        return self

    def predict_proba(self, X):
        device = next(self.model.parameters()).device
        self.model.eval()
        with torch.no_grad():
            prob = torch.sigmoid(
                self.model(torch.tensor(X, dtype=torch.float32).to(device))
            ).cpu().numpy()
        return np.column_stack([1 - prob, prob])   # shape: (n, 2)

    def __repr__(self):
        return "TradingNN(residual)"

In [ ]:
models = [
    LogisticRegression(), 
    SVC(kernel='poly', probability=True), 
    XGBClassifier(
        max_depth=3,              # ağaç derinliğini sınırla (default 6)
        n_estimators=200,         # ağaç sayısı
        learning_rate=0.05,       # küçük learning rate
        subsample=0.8,            # her ağaçta verinin %80'i
        colsample_bytree=0.8,    # her ağaçta feature'ların %80'i
        reg_alpha=1.0,            # L1 regularization
        reg_lambda=5.0,           # L2 regularization
        min_child_weight=5,       # yapraklarda min örnek sayısı
        gamma=0.3,                # split için min kayıp azalması
        early_stopping_rounds=20, # erken durdurma
        eval_metric='auc'
    ),
    TradingNNWrapper(epochs=150, batch_size=128, lr=1e-3, patience=200)
]

In [ ]:
for model in models:
    if isinstance(model, XGBClassifier):
        model.fit(X_train, Y_train, eval_set=[(X_valid, Y_valid)], verbose=False)
    elif isinstance(model, TradingNNWrapper):
        model.fit(X_train, Y_train, X_val=X_valid, y_val=Y_valid)  # val set gerekiyor
    else:
        model.fit(X_train, Y_train)

    print(f'{model} :')
    print('Train AUC :', metrics.roc_auc_score(Y_train, model.predict_proba(X_train)[:,1]))
    print('Val AUC   :', metrics.roc_auc_score(Y_valid, model.predict_proba(X_valid)[:,1]))
    print()

In [ ]:
ConfusionMatrixDisplay.from_estimator(models[2], X_valid, Y_valid)
plt.show()

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn import metrics

# ── Feature Grupları ──────────────────────────────────────────────────────────
FEATURE_GROUPS = {
    "technical": [
        'open-close', 'low-high', 'is_quarter_end',
        'SMA_7', 'SMA_21', 'SMA_50',
        'MACD', 'MACD_signal', 'MACD_hist',
        'RSI_14', 'BB_width', 'BB_upper', 'BB_lower',
        'ATR_14', 'Stoch_K', 'Stoch_D',
        'OBV', 'Momentum_10', 'VWAP',
    ],
    "sentiment": [
        'avg_sentiment', 'news_count', 'sentiment_7d',
        'sentiment_14d', 'sentiment_30d',
    ],
    "macro_fx": [
        'USD_change', 'USD_change_5d', 'USD_sma_7',
        'close_usd', 'close_usd_change',
        'TP_DK_USD_S_YTL', 'TP_DK_USD_A_YTL',
        'USD_spread', 'USD_volatility', 'EUR_USD_ratio',
    ],
    "macro_other": [
        'TP_FG_J0', 'TP_FG_J01', 'TP_FG_J02',
        'TP_AB_B1', 'TP_AB_B3',
    ],
    "fred": [
        'Fed Faiz Oranı', '10Y Tahvil Faizi', '2Y Tahvil Faizi',
        'Yield Curve (10Y-2Y spread)', 'VIX (volatilite endeksi)',
        'Ham Petrol Fiyatı', 'USD/CNY', 'Tüketici Güven Endeksi',
        'TED Spread (kredi riski)',
    ],
}

# ── Test edilecek kombinasyonlar ──────────────────────────────────────────────
COMBINATIONS = {
    "only_technical":             ["technical"],
    "technical+sentiment":        ["technical", "sentiment"],
    "technical+macro_fx":         ["technical", "macro_fx"],
    "technical+fred":             ["technical", "fred"],
    "technical+sentiment+macro":  ["technical", "sentiment", "macro_fx", "macro_other"],
    "all":                        ["technical", "sentiment", "macro_fx", "macro_other", "fred"],
}


def run_nn_experiments(
    df: pd.DataFrame,
    target_col: str = "target",
    combinations: dict = COMBINATIONS,
    nn_kwargs: dict = None,
    test_size: float = 0.1,
    random_state: int = 42,
) -> pd.DataFrame:
    """
    Farklı feature kombinasyonlarını TradingNNWrapper ile dener.

    Returns
    -------
    pd.DataFrame — her kombinasyon için Train/Val AUC sonuçları
    """
    if nn_kwargs is None:
        nn_kwargs = dict(epochs=50, batch_size=128, lr=1e-3, patience=15)

    results = []

    for combo_name, groups in combinations.items():
        # İlgili feature sütunlarını al (df'de yoksa atla)
        cols = []
        for g in groups:
            for c in FEATURE_GROUPS[g]:
                if c in df.columns:
                    cols.append(c)
                else:
                    print(f"  ⚠️  Eksik sütun: {c} ({combo_name})")

        if not cols:
            print(f"  ❌ {combo_name}: hiç sütun bulunamadı, atlanıyor.")
            continue

        print(f"\n{'='*55}")
        print(f"▶  {combo_name}  ({len(cols)} feature)")
        print(f"{'='*55}")

        X = df[cols].values
        y = df[target_col].values

        scaler = StandardScaler()
        X = scaler.fit_transform(X)

        X_train, X_valid, Y_train, Y_valid = train_test_split(
            X, y,
            test_size=test_size,
            random_state=random_state,
            stratify=y,
        )

        try:
            wrapper = TradingNNWrapper(**nn_kwargs)
            wrapper.fit(X_train, Y_train, X_val=X_valid, y_val=Y_valid)

            train_auc = metrics.roc_auc_score(Y_train, wrapper.predict_proba(X_train)[:, 1])
            val_auc   = metrics.roc_auc_score(Y_valid, wrapper.predict_proba(X_valid)[:, 1])

            print(f"  Train AUC: {train_auc:.4f} | Val AUC: {val_auc:.4f}")

            results.append({
                "combination":  combo_name,
                "n_features":   len(cols),
                "train_auc":    round(train_auc, 4),
                "val_auc":      round(val_auc, 4),
            })
        except Exception as e:
            print(f"  ❌ Hata: {e}")
            results.append({
                "combination": combo_name,
                "n_features":  len(cols),
                "train_auc":   None,
                "val_auc":     None,
            })

    result_df = pd.DataFrame(results).sort_values("val_auc", ascending=False)
    print(f"\n{'='*55}")
    print("📊 SONUÇLAR (Val AUC'ye göre sıralı):")
    print(result_df.to_string(index=False))
    return result_df


In [ ]:
# Hızlı test (az epoch)
results = run_nn_experiments(df, nn_kwargs=dict(epochs=50, batch_size=128, lr=1e-3, patience=20))



In [ ]:
DB_DIR = os.path.join(
    '/Users/aliemre2023/Desktop/apache_project/bist100_airflow/src/db'
)
DB_PATH = os.path.join(DB_DIR, 'bist.db')

def get_connection() -> sqlite3.Connection:
    os.makedirs(DB_DIR, exist_ok=True)
    conn = sqlite3.connect(DB_PATH)
    conn.row_factory = sqlite3.Row
    conn.execute("PRAGMA foreign_keys = ON")
    return conn

conn = get_connection()

In [ ]:
import pandas as pd

SIRKET_CODE = "HLGYO"  # buraya istediğin şirket kodunu yaz

query = """
SELECT n.news_date AS date,
       n.news_ratio AS score,
       n.news_type AS type
FROM   news n
JOIN   news_sirket ns ON ns.news_id  = n.news_id
JOIN   sirket      s  ON s.sirket_id = ns.sirket_id
WHERE  s.sirket_code = ?
ORDER BY n.news_date
"""

df = pd.read_sql_query(query, conn, params=(SIRKET_CODE,))
df

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# 1) Tarihe göre ortalama sentiment skoru
df["date"] = pd.to_datetime(df["date"])
daily_sentiment = df.groupby("date")["score"].mean().reset_index()

# 2) yfinance'dan aynı tarih aralığında hisse verisini çek
start = daily_sentiment["date"].min().strftime("%Y-%m-%d")
end   = (daily_sentiment["date"].max() + pd.Timedelta(days=1)).strftime("%Y-%m-%d")
ticker = yf.Ticker(f"{SIRKET_CODE}.IS")
hist = ticker.history(start=start, end=end)
hist.index = hist.index.tz_localize(None)

# 3) Matplotlib karşılaştırma grafiği
fig, ax1 = plt.subplots(figsize=(12, 5))

color_price = "tab:blue"
ax1.set_xlabel("Tarih")
ax1.set_ylabel("Hisse Fiyatı (₺)", color=color_price)
ax1.plot(hist.index, hist["Close"], color=color_price, marker="o", label="Kapanış Fiyatı")
ax1.tick_params(axis="y", labelcolor=color_price)

ax2 = ax1.twinx()
color_sent = "tab:red"
ax2.set_ylabel("Ortalama Sentiment Skoru", color=color_sent)
ax2.bar(daily_sentiment["date"], daily_sentiment["score"], color=color_sent, alpha=0.5, width=0.6, label="Sentiment")
ax2.tick_params(axis="y", labelcolor=color_sent)
ax2.axhline(y=0, color="gray", linestyle="--", linewidth=0.8)

ax1.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m-%d"))
fig.autofmt_xdate()

fig.suptitle(f"{SIRKET_CODE} — Hisse Fiyatı vs Sentiment Skoru", fontsize=14)
fig.tight_layout()
plt.show()